# 🚀 VLA 연구 · 3회차 실습
## 개발환경 구축 & Python 보완

| 항목 | 내용 |
|------|------|
| 오늘 목표 | Colab Pro Plus + GCS 연동, NumPy·Matplotlib·PIL 핵심 실습 |
| 미니과제 | 택배 시뮬레이터 (격자 시각화) |
| 제출 방법 | 완성된 노트북을 GCS 공유 폴더에 저장 후 조교에게 링크 공유 |

> **이 노트북을 여는 순서**  
> 런타임 → 런타임 유형 변경 → **GPU (T4)** 선택 후 저장

---
## SECTION 0  |  환경 확인


>시각화 시 사용할 한글 폰트 설정 (아래 블록 실행 후 **런타임 재시작**)

In [ ]:
# 한글 폰트 설정 (Colab 환경)
import matplotlib
import matplotlib.font_manager as fm

## Colab에 나눔폰트 설치
import subprocess
subprocess.run(['apt-get', 'install', '-y', 'fonts-nanum'], capture_output=True)
fm._load_fontmanager(try_read_cache=False)  # 폰트 캐시 갱신

## 나눔고딕 적용
matplotlib.rcParams['font.family'] = 'NanumGothic'
matplotlib.rcParams['axes.unicode_minus'] = False  # 마이너스 기호 깨짐 방지

### 0-1. GPU 및 Python 환경 확인

아래 셀을 실행하면 현재 할당된 GPU 정보와 Python 버전을 확인할 수 있습니다.  
T4 또는 L4가 보이면 정상입니다. **'No GPU'가 뜨면 런타임 유형을 GPU로 바꿔주세요.**

In [ ]:
import torch
import sys

print('=== 환경 정보 ===')
print(f'Python: {sys.version.split()[0]}')
print(f'PyTorch: {torch.__version__}')

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU 메모리: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠ GPU 없음 — 런타임 유형을 GPU로 변경하세요')

---
## SECTION 1  |  Google Drive 마운트

### 1-1. Google Drive 연결

개인 Drive를 마운트합니다. 팀 공유 파일은 GCS를 사용하지만,  
노트북 작업 파일의 임시 저장·백업용으로 Drive를 함께 활용합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_BASE = '/content/drive/MyDrive/VLA_2026'
os.makedirs(DRIVE_BASE, exist_ok=True)
print(f'Drive 마운트 완료: {DRIVE_BASE}')

---
## SECTION 2  |  Google Cloud Storage 연동

### 2-1. GCS 인증 및 버킷 연결

팀 공유 버킷에 접근합니다. 아래 `BUCKET_NAME`을 조교에게 받은 버킷 이름으로 바꿔주세요.

**처음 실행 시**: 팝업 창에서 Google 계정으로 로그인 후 권한을 허용하세요.

In [ ]:
# google-cloud-storage 패키지 확인 및 설치
# Colab에는 보통 설치되어 있지만, 버전 문제가 생기면 아래 주석을 해제하세요
# !pip install --quiet google-cloud-storage

import google.cloud.storage
print(f'google-cloud-storage 버전: {google.cloud.storage.__version__}')

In [ ]:
from google.colab import auth
auth.authenticate_user()
print('✓ Google 인증 완료')

In [ ]:
# ── Google Cloud 프로젝트 ID 설정 ────────────────────────
# console.cloud.google.com 에서 프로젝트 이름 옆 ID를 확인하세요
PROJECT_ID = 'vla-research-2026'
# ──────────────────────────────────────────────────────

!gcloud config set project {PROJECT_ID}
print(f'✓ 프로젝트 설정 완료: {PROJECT_ID}')

In [ ]:
# ── 버킷 이름을 아래에 입력하세요 ───────────────────────
BUCKET_NAME = 'vla-team-daejeon-2026'
# ──────────────────────────────────────────────────────

# 팀 공유 폴더 구조
GCS_PATHS = {
    'trajectories' : f'gs://{BUCKET_NAME}/trajectories/',
    'checkpoints'  : f'gs://{BUCKET_NAME}/checkpoints/',
    'results'      : f'gs://{BUCKET_NAME}/results/',
    'demos'        : f'gs://{BUCKET_NAME}/demos/',
    'notebooks'    : f'gs://{BUCKET_NAME}/notebooks/',
}

print('=== 팀 GCS 경로 ===')
for k, v in GCS_PATHS.items():
    print(f'  {k:15s}: {v}')

### 2-2. GCS 접근 테스트

버킷에 테스트 파일을 올리고 내려받아 연동이 정상인지 확인합니다.

In [ ]:
# 버킷 접근 가능 여부 확인
# 아래 출력에 버킷 이름이 보이면 정상입니다.
# 오류가 나면 하단 '흔한 오류' 마크다운을 참고하세요.
!gsutil ls gs://{BUCKET_NAME}/
print(f'✓ 버킷 [{BUCKET_NAME}] 접근 성공')

### ⚠️ GCS 연결 시 흔한 오류와 해결법

| 오류 메시지 | 원인 | 해결 방법 |
|-------------|------|----------|
| `AccessDeniedException: 403` | 권한 없음 | 조교에게 계정 이메일을 알려주고 IAM 권한 추가 요청 |
| `BucketNotFoundException: 404` | 버킷 이름 오타 | `BUCKET_NAME` 변수를 다시 확인 |
| `Invalid project id` | 프로젝트 ID 오타 | console.cloud.google.com에서 정확한 ID 복사 |
| `ServiceException: 401` | 인증 만료 | `auth.authenticate_user()` 셀을 다시 실행 |
| `CommandException: No URLs matched` | 경로 오타 | `gs://` 로 시작하는지, 버킷명 확인 |

> **그래도 안 되면**: 오류 메시지 전체를 Claude Code에 붙여넣으세요.

In [ ]:
import json
from datetime import datetime

# 테스트 파일 업로드
test_data = {
    'student': '본인 이름을 여기에 입력하세요',
    'timestamp': datetime.now().isoformat(),
    'message': 'GCS 연동 성공!'
}

# 파일명의 'name'을 각자 이름으로 변경
file_name = 'gcs_test_name.json'
file_path = '/tmp/'+file_name
with open(file_path, 'w') as f:
    json.dump(test_data, f, ensure_ascii=False, indent=2)

!gsutil cp {file_path} {GCS_PATHS['notebooks']}
print('✓ 업로드 완료')

# 목록 확인
print('\n버킷 내 파일 목록:')
!gsutil ls {GCS_PATHS['notebooks']}

In [ ]:
# 다운로드 테스트
!gsutil cp {GCS_PATHS['notebooks']}{file_name} /tmp/gcs_downloaded.json

with open('/tmp/gcs_downloaded.json') as f:
    downloaded = json.load(f)

print('✓ 다운로드 완료')
print(json.dumps(downloaded, ensure_ascii=False, indent=2))

### 2-3. GCS 편의 함수

매번 `!gsutil` 명령을 치는 대신 Python 함수로 편하게 사용합니다.  
이후 회차에서도 이 함수들을 재사용합니다.

In [ ]:
from google.cloud import storage
import os

def gcs_upload(local_path: str, gcs_folder: str):
    """로컬 파일을 GCS 폴더에 업로드"""
    client = storage.Client()
    bucket = client.bucket(BUCKET_NAME)
    filename = os.path.basename(local_path)
    blob = bucket.blob(gcs_folder.replace(f'gs://{BUCKET_NAME}/', '') + filename)
    blob.upload_from_filename(local_path)
    print(f'✓ 업로드: {local_path} → {gcs_folder}{filename}')

def gcs_download(gcs_path: str, local_dir: str = '/tmp/'):
    """GCS 파일을 로컬에 다운로드"""
    client = storage.Client()
    bucket = client.bucket(BUCKET_NAME)
    blob_path = gcs_path.replace(f'gs://{BUCKET_NAME}/', '')
    filename = os.path.basename(blob_path)
    local_path = os.path.join(local_dir, filename)
    bucket.blob(blob_path).download_to_filename(local_path)
    print(f'✓ 다운로드: {gcs_path} → {local_path}')
    return local_path

def gcs_list(gcs_folder: str):
    """GCS 폴더의 파일 목록 반환"""
    client = storage.Client()
    bucket = client.bucket(BUCKET_NAME)
    prefix = gcs_folder.replace(f'gs://{BUCKET_NAME}/', '')
    blobs = list(bucket.list_blobs(prefix=prefix))
    return [b.name for b in blobs]

print('✓ GCS 편의 함수 등록 완료')

---
## SECTION 3  |  Claude Code 활용법 (5분)

### 3-1. 막혔을 때 Claude Code 사용하는 법

실습 중 오류가 나거나 막히면 **Claude Code에 먼저 물어보세요**.  
조교에게 질문하기 전에 Claude Code를 먼저 시도하는 것이 기본입니다.

**효과적인 질문 방법:**
```
# 나쁜 예 ❌
"코드가 안 돼요"

# 좋은 예 ✅
"아래 코드에서 다음 오류가 발생합니다.
[코드 붙여넣기]
[오류 메시지 붙여넣기]
어떻게 고치면 될까요?"
```

아래 셀을 일부러 오류 내고, Claude Code로 해결해 보는 연습을 합니다.

In [ ]:
# 아래 코드에는 의도적인 오류가 있습니다.
# 오류 메시지를 Claude Code에 붙여넣고 고쳐 보세요.

import numpy as np

arr = np.array([1, 2, 3, 4, 5])
print(arr.shpe)          # 오류 1: 오타
print(arr[10])            # 오류 2: 인덱스 초과

> **연습**: 위 셀의 오류 메시지를 Claude Code에 붙여넣고 수정 방법을 물어보세요.
> 고친 코드를 아래 셀에 작성합니다.

> [참고] 일반 텍스트로 붙여넣기 : ctrl+shift+v

In [ ]:
# ✏️ 오류를 수정한 코드를 여기에 작성하세요


---
## SECTION 4  |  NumPy 기초

### 4-1. 배열 생성과 기본 속성

NumPy 배열은 이후 이미지 데이터, trajectory 데이터를 다루는 **모든 실습의 기반**입니다.  
MuJoCo 렌더링 결과도 `(H, W, 3)` shape의 NumPy 배열로 돌아옵니다.

In [ ]:
import numpy as np

# 1D 배열
a = np.array([1, 2, 3, 4, 5])
print('1D:', a)
print('shape:', a.shape, '| dtype:', a.dtype)

# 2D 배열 (행렬)
b = np.array([[1, 2, 3],
              [4, 5, 6]])
print('\n2D:\n', b)
print('shape:', b.shape)  # (행, 열)

# 3D 배열 — 이미지 형태 (세로, 가로, 채널)
img_like = np.zeros((100, 100, 3), dtype=np.uint8)  # H=100, W=100, C=3(RGB)
print('\n이미지 형태 배열 shape:', img_like.shape)

### 4-2. 배열 생성 함수

In [ ]:
# 자주 쓰는 배열 생성 함수
print('zeros:', np.zeros((2, 3))) # 괄호 주의! (())
print()
print('ones:', np.ones((2, 3)))   # 괄호 주의! (())
print()
print('arange:', np.arange(0, 10, 2))   # 0부터 10 미만까지 2 간격으로 생성
print()
print('linspace:', np.linspace(0, 1, 5)) # 0~1 구간을 4등분한 경계값 5개 생성
print()
print('random:', np.random.rand(2, 3))   # 0~1 균일분포 랜덤 숫자를 2행 3열 배열로 나열
print()
print('random int:', np.random.randint(0, 256, size=(2, 3)))  # 0~255 정수 랜덤 숫자를 2행 3열 배열로 나열

### 4-3. 슬라이싱 & 인덱싱

이미지에서 특정 영역이나 채널을 추출할 때 매우 자주 씁니다.

In [ ]:
arr = np.arange(1, 26).reshape(5, 5)  # 1~25를 5×5로 변환
print('원본:\n', arr)

print('\n첫 번째 행:', arr[0])
print('마지막 열:', arr[:, -1])
print('\n2~4행, 1~3열:\n', arr[1:4, 0:3])

# 이미지에서 R 채널만 추출하는 것과 같은 패턴
img = np.random.randint(0, 256, (100, 100, 3), dtype=np.uint8)
R_channel = img[:, :, 0]   # 빨강
G_channel = img[:, :, 1]   # 초록
B_channel = img[:, :, 2]   # 파랑
print(f'\nRGB 이미지 shape: {img.shape}')
print(f'R 채널 shape: {R_channel.shape}')

### 4-4. 브로드캐스팅

shape이 다른 배열끼리 연산할 때 NumPy가 자동으로 크기를 맞춥니다.  
이미지 정규화(0\~255 → 0\~1)에서 바로 사용됩니다.

> 브로드캐스팅 규칙
>
> 두 배열의 shape을 오른쪽 끝부터 비교해서, 각 차원 쌍이 다음 중 하나면 됩니다.
> * 같다 → OK
> * 둘 중 하나가 1이다 → 1인 쪽을 늘려서 맞춤
> * 둘 다 1도 아니고 다르다 → 오류

In [ ]:
img = np.random.randint(0, 256, (4, 4, 3), dtype=np.uint8)
print('원본 이미지 (일부):\n', img[:2, :2])

# 정규화: 모든 픽셀값을 255로 나누기
img_norm = img / 255.0
print('\n정규화 후 (일부):\n', img_norm[:2, :2].round(3))
print('값 범위: {:.3f} ~ {:.3f}'.format(img_norm.min(), img_norm.max()))

# 채널별 평균값 계산 (브로드캐스팅 활용)
channel_mean = img.mean(axis=(0, 1))  # H와 W를 동시에 평균
print('\n채널별 평균값 (R, G, B):', channel_mean.round(1))

### 4-5. 연습문제 ✏️

아래 문제를 풀어보세요.
* 우선 스스로 작성해 보세요.
* 막힌다면 충분히 고민한 이후에 Claude Code에 질문하세요.

In [ ]:
# 문제 1
# shape이 (50, 50, 3)인 무작위 uint8 이미지 배열을 만들고
# R·G·B 채널 각각의 평균값을 출력하세요.

# 여기에 코드 작성

In [ ]:
# 문제 2
# 위에서 만든 이미지에서 R 채널 값이 200 이상인 픽셀의 개수를 구하세요.
# 힌트: 불리언 마스킹 → (img[:,:,0] >= 200)

# 여기에 코드 작성

In [ ]:
# 문제 3
# shape이 (3, 4) 인 배열 A와 shape이 (3,) 인 배열 B를 만들고
# A의 각 행에서 B를 빼는 연산을 수행하세요. 결과 shape을 확인하세요.
# 힌트: B를 (3,1)로 reshape 해야 합니다

# 여기에 코드 작성

---
## SECTION 5  |  Matplotlib 기초

### 5-1. 기본 플롯

실험 결과 시각화와 MuJoCo 렌더링 이미지 확인에 필수입니다.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 기본 선 그래프
x = np.linspace(0, 2 * np.pi, 100)
fig, axes = plt.subplots(1, 2, figsize=(10, 4)) # 그래프를 넣을 칸의 형태와 이미지 크기를 지정

## axes[0] : 왼쪽 그래프 — 삼각함수 선 그래프
axes[0].plot(x, np.sin(x), label='sin(x)', color='royalblue')
axes[0].plot(x, np.cos(x), label='cos(x)', color='tomato', linestyle='--')
axes[0].set_title('삼각함수')
axes[0].set_xlabel('x')
axes[0].legend() # label로 지정한 이름을 범례로 띄움
axes[0].grid(True, alpha=0.3) # 격자선 추가 (alpha는 투명도)

## axes[1] : 오른쪽 그래프 — 히스토그램
data = np.random.randn(1000)
axes[1].hist(data, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].set_title('정규분포 히스토그램')
axes[1].set_xlabel('값')
axes[1].set_ylabel('빈도')

plt.tight_layout() # 그래프 사이 간격과 여백을 자동으로 정리
plt.show() # 최종 출력

### 5-2. 이미지 시각화

MuJoCo에서 얻은 이미지를 확인할 때 가장 많이 쓰는 패턴입니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 가상의 RGB 이미지 생성 (나중에 MuJoCo 렌더링 결과로 대체)
np.random.seed(42) # 난수 생성기가 숫자를 뽑는 출발점을 고정 (=> 항상 동일한 값 나옴)
fake_img = np.random.randint(0, 256, (100, 100, 3), dtype=np.uint8)

# RGB 채널 분리
R = fake_img[:, :, 0]
G = fake_img[:, :, 1]
B = fake_img[:, :, 2]

fig, axes = plt.subplots(1, 4, figsize=(14, 4))

axes[0].imshow(fake_img)
axes[0].set_title('원본 RGB')
axes[0].axis('off')

# zip : 여러 리스트를 같은 인덱스끼리 묶어주는 함수
for ax, ch, name, cmap in zip(axes[1:], [R, G, B], ['R', 'G', 'B'], ['Reds', 'Greens', 'Blues']):
    ax.imshow(ch, cmap=cmap, vmin=0, vmax=255)
    ax.set_title(f'{name} 채널  (평균: {ch.mean():.1f})')
    ax.axis('off')

"""
# zip을 쓰지 않으면
channels = [R, G, B]
names = ['R', 'G', 'B']
cmaps = ['Reds', 'Greens', 'Blues']

for i in range(3):
    axes[i+1].imshow(channels[i], cmap=cmaps[i], vmin=0, vmax=255)
    axes[i+1].set_title(f'{names[i]} 채널')
    axes[i+1].axis('off')
"""

plt.suptitle('RGB 채널 분리 시각화', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

### 5-3. PIL 기초 — 실제 이미지 파일 다루기

PNG/JPG 파일을 읽고 NumPy 배열로 변환하는 패턴입니다.  
8회차부터는 MuJoCo가 렌더링한 이미지를 이 방식으로 저장·불러옵니다.

In [ ]:
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

# 테스트용 이미지 생성 및 저장
test_img = gcs_download(f"{GCS_PATHS['notebooks']}test_img.png")

# PIL로 불러오기
loaded = Image.open(test_img)
print(f'PIL 이미지: size={loaded.size}, mode={loaded.mode}')

# NumPy 배열로 변환
arr = np.array(loaded)
print(f'NumPy 배열: shape={arr.shape}, dtype={arr.dtype}')

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(arr)
axes[0].set_title('불러온 이미지')
axes[0].axis('off')

# 그레이스케일 변환
gray = np.mean(arr, axis=2).astype(np.uint8)
axes[1].imshow(gray, cmap='gray')
axes[1].set_title('그레이스케일 변환')
axes[1].axis('off')

plt.tight_layout()
plt.show()

### 5-4. 연습문제 ✏️

In [ ]:
# 문제: 아래 데이터를 막대 그래프로 시각화하세요. (matplot bar 함수 활용)
# 이 데이터는 나중에 '씬 다양성 리포트' (9회차 미니과제)에서 실제로 사용합니다.

labels = ['pick', 'push', 'reorder', 'ignore']
counts = [42, 28, 19, 11]

# 요구사항:
# 1. 각 막대에 다른 색상을 적용하세요
# 2. 각 막대 위에 해당 개수(숫자)를 표시하세요
# 3. y축 레이블은 '씬 수', 제목은 '행동 카테고리 분포'로 설정하세요

# 여기에 코드 작성


---
## SECTION 6  |  미니과제 — 택배 시뮬레이터

## 🎯 택배 시뮬레이터

### 과제 설명

NumPy로 **10×10 격자**를 만들고, 각 셀 상태를 색상으로 시각화하는 함수를 작성하세요.

| 상태 | 의미 | 색상 |
|------|------|------|
| `blocked` | 앞에 장애물이 있어 접근 불가 | 빨강 |
| `accessible` | 바로 집을 수 있음 | 초록 |
| `empty` | 빈 칸 | 흰색 |

**요구사항:**
- 함수를 호출할 때마다 무작위로 다른 배치가 생성되어야 합니다
- 격자 셀 중앙에 상태 이름 첫 글자를 텍스트로 표시하세요 (`B`, `A`, `E`)
- 격자 아래에 각 상태의 개수를 출력하세요

### 연구와의 연결
> 이 격자는 우리 VLA 연구에서 MuJoCo 환경의 **단순화된 추상 표현**입니다.  
> 나중에 VLM이 'accessible한 박스를 pick하고, blocked된 박스 앞의 장애물을 push하라'고 판단하는 구조가 됩니다.

### 힌트 블록

막히면 아래 힌트를 순서대로 열어보세요. (전부 보지 말고 필요한 만큼만!)

**힌트 1 — 격자 데이터 구조**  
```python
# 0=empty, 1=accessible, 2=blocked 으로 인코딩
grid = np.random.choice([0, 1, 2], size=(10, 10), p=[0.4, 0.35, 0.25])
```

**힌트 2 — 색상 매핑**  
```python
color_map = {0: 'white', 1: 'mediumseagreen', 2: 'tomato'}
label_map = {0: 'E', 1: 'A', 2: 'B'}
```

**힌트 3 — 격자 그리기 기본 패턴**  
```python
fig, ax = plt.subplots(figsize=(7, 7))
for row in range(10):
    for col in range(10):
        state = grid[row, col]
        rect = plt.Rectangle([col, 9-row], 1, 1,
                             facecolor=color_map[state],
                             edgecolor='gray', linewidth=0.8)
        ax.add_patch(rect)
        ax.text(col+0.5, 9-row+0.5, label_map[state],
                ha='center', va='center', fontsize=9)
```

**힌트 4 — 축 정리**  
```python
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_aspect('equal')
ax.axis('off')
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

def simulate_delivery_grid(seed=None):
    """
    택배 시뮬레이터: 10×10 격자에 accessible/blocked/empty 상태를 무작위로 배치하고 시각화.

    Parameters
    ----------
    seed : int, optional
        재현 가능성을 위한 랜덤 시드 (None이면 매번 다른 배치)
    """
    # ── 여기에 코드를 작성하세요 ──────────────────────────────
    pass
    # ─────────────────────────────────────────────
    return grid


# 실행 테스트
simulate_delivery_grid()           # 무작위 배치

In [ ]:
# 재현성 테스트: 같은 seed를 넣으면 같은 결과가 나와야 합니다
print('=== seed=42 (두 실행이 동일해야 함) ===')
g1 = simulate_delivery_grid(seed=42)
g2 = simulate_delivery_grid(seed=42)

print('=== seed=None (두 실행이 달라야 함) ===')
g3 = simulate_delivery_grid()
g4 = simulate_delivery_grid()

print('동일 seed → 동일 결과:', np.array_equal(g1, g2))
print('seed 없음 → 다른 결과:', not np.array_equal(g3, g4))

In [ ]:
# 도전 과제 (선택): 특정 셀을 '목표 택배(target)'로 지정하고
# 해당 셀을 파란색 테두리로 강조해 보세요. (hint : edgecolor='royalblue')

# 여기에 코드 작성 (선택)

---
## SECTION 7  |  결과 저장 & 제출

### 7-1. 완성된 노트북을 GCS에 저장하기

미니과제를 완성했으면 아래 셀을 실행하여 GCS에 저장하고 조교에게 경로를 공유하세요.

In [ ]:
# 현재 노트북을 Drive에 저장한 뒤 GCS로 복사
import subprocess
from datetime import datetime

# 본인 이름으로 변경하세요
STUDENT_NAME = '본인이름'   # ← 변경 필요

timestamp = datetime.now().strftime('%m%d_%H%M')
notebook_name = f'session3_{STUDENT_NAME}_{timestamp}.ipynb'

# Colab 현재 노트북 파일 경로
# (파일 → Drive에 사본 저장 후 아래 경로를 실제 경로로 변경)
LOCAL_NB_PATH = f'/content/drive/MyDrive/VLA_2026/{notebook_name}'

# GCS에 업로드
gcs_target = f'{GCS_PATHS["notebooks"]}{notebook_name}'
!gsutil cp "{LOCAL_NB_PATH}" "{gcs_target}"

print(f'\n✓ 제출 완료!')
print(f'  GCS 경로: {gcs_target}')
print(f'  이 경로를 조교에게 공유하세요.')

---
## SECTION 8  |  오늘 배운 것 정리

### 오늘 다룬 내용

| 항목 | 확인 |
|------|------|
| Colab Pro Plus GPU 연결 확인 | ☐ |
| Google Drive 마운트 | ☐ |
| GCS 버킷 연동 + 업로드/다운로드 테스트 | ☐ |
| Claude Code로 오류 해결 연습 | ☐ |
| NumPy: 배열 생성·슬라이싱·브로드캐스팅 | ☐ |
| Matplotlib: 선 그래프·이미지 시각화 | ☐ |
| PIL: 이미지 파일 ↔ NumPy 배열 변환 | ☐ |
| 미니과제: 택배 시뮬레이터 완성 | ☐ |

### 다음 회차 예고 (4회차)

- **PyTorch**: Tensor, Autograd, 학습 루프
- **미니과제**: 내 손글씨 인식기 — MNIST로 학습한 모델에 직접 찍은 숫자 이미지 넣어보기

> 4회차 수업 전날 조교가 Colab 파일 링크를 공유합니다.